In [1]:
try:
    import kagglehub
except ImportError:
    !pip install kagglehub
try:
    import huggingface_hub
except ImportError:
    !pip install huggingface_hub
try:
    import ipywidgets
except ImportError:
    !pip install ipywidgets
    !pip install --upgrade ipywidgets

!pip install --upgrade notebook jupyterlab

import os
import sys
if os.path.join(os.getcwd(), '..') not in sys.path:
    sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.Unet.Unet_model import UNet
from src.utils.ds_check import check_ds
from src.dataloaders.dataset00 import MyDataset
from src.utils.logger import SemanticSegmentationLogger


In [2]:
from torch.utils.data import random_split, DataLoader
import torch
from torchvision import transforms
import numpy as np

ds_path = check_ds(
    "Drone Seg Dataset",
    "santurini/semantic-segmentation-drone-dataset",
)

print("Dataset downloaded, path = ", ds_path)

dataset = MyDataset(
    image_dir="dataset/Drone Seg Dataset/classes_dataset/classes_dataset/original_images",
    label_dir="dataset/Drone Seg Dataset/classes_dataset/classes_dataset/label_images_semantic",
    transform=transforms.Compose([
        # 1. 将 PIL Image 或 numpy 数组转为 Tensor (自动将 [0, 255] 转为 [0.0, 1.0] 的 FloatTensor)
        # 注意：ToTensor() 会自动完成除以 255 的操作，并且会将维度从 HWC 变为 CHW
        transforms.ToTensor(), 
        
        # (可选) 如果你需要进一步标准化 (例如减去均值除以方差)，可以在这里加 Normalize
        # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]), 
)

train_set, test_set = random_split(dataset, [int(0.9 * len(dataset)), len(dataset) - int(0.9 * len(dataset))])
train_loader = DataLoader(dataset=train_set, batch_size=4, shuffle=True)
test_loader = DataLoader(dataset=test_set, batch_size=4, shuffle=False)

model = UNet(num_classes=5)

print(f"数据集总样本数: {len(dataset)}")
print(f"训练集样本数: {len(train_set)}")
print(f"测试集样本数: {len(test_set)}")

img0, lbl0, _ = train_set.__getitem__(0);
print(f"Train set - img shape: {img0.shape}, lbl shape: {lbl0.shape}")
lbl_unique = torch.unique(lbl0)
print(f"Train set - unique labels in lbl0: {lbl_unique}")

img0, lbl0, _ = test_set.__getitem__(0);
print(f"Test set - img shape: {img0.shape}, lbl shape: {lbl0.shape}")
lbl_unique = torch.unique(lbl0)
print(f"Train set - unique labels in lbl0: {lbl_unique}")

img0, lbl0, _ = dataset.__getitem__(0)

print("图像1的形状:", img0.shape)
print("标签1的形状:", lbl0.shape)

img0_batched = img0.unsqueeze(0)
ans0 = model(img0_batched)  # 添加批次维度
print("模型输出的形状:", ans0.shape)

sample_output = torch.argmax(ans0, dim=1).detach().cpu().numpy()
print("模型输出的唯一值 (类别索引):", np.unique(sample_output))

print("Train logger: Unet_Segmentation_Logger")
logger = SemanticSegmentationLogger("Unet_Segmentation_Logger")

# print("Model summary: ")
# print(model)

Drone Seg Dataset 数据集已存在，路径: /home/jovyan/2026.3.17/resources/dataset/Drone Seg Dataset
Dataset downloaded, path =  /home/jovyan/2026.3.17/resources/dataset/Drone Seg Dataset
成功匹配文件: 463.png, 175.png, 535.png 等 400 个文件
数据集总样本数: 400
训练集样本数: 360
测试集样本数: 40
Train set - img shape: torch.Size([3, 736, 960]), lbl shape: torch.Size([1, 736, 960])
Train set - unique labels in lbl0: tensor([0, 2, 3, 4])
Test set - img shape: torch.Size([3, 736, 960]), lbl shape: torch.Size([1, 736, 960])
Train set - unique labels in lbl0: tensor([0, 1, 2, 3, 4])
图像1的形状: torch.Size([3, 736, 960])
标签1的形状: torch.Size([1, 736, 960])
模型输出的形状: torch.Size([1, 5, 736, 960])
模型输出的唯一值 (类别索引): [0 1 2 3 4]
Train logger: Unet_Segmentation_Logger


In [3]:
import torch
from torch import nn, optim
import numpy as np
import os

# 检查是否有可用的GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

import time
print(time.time())

Using device: cuda
1774347040.3545527


In [4]:
import time
import sys

def train_phase(model, dataloader, criterion, optimizer, num_epochs, num_classes, phase_id, logger, sample_interval=50):
    """
    训练一个阶段。
    
    Args:
        model: 已构建的模型实例。
        dataloader: DataLoader实例，提供训练数据。
        criterion: 损失函数。
        optimizer: 优化器。
        num_epochs: 此阶段的训练轮数。
        num_classes: 数据集的类别总数。
        phase_id: 阶段ID。
        logger: 日志记录器实例。
        sample_interval: 每隔多少个epoch保存一次样本。
    """
    since = time.time()
    last_loss = None
    patience = 0
    loss_threshold = 1e-5
    max_duration_hours = 10
    max_duration_seconds = max_duration_hours * 3600

    print(f"[Phase {phase_id}] Starting training for {num_epochs} epochs with lr={optimizer.param_groups[0]['lr']:.6f}...")
    print(f"[Phase {phase_id}] Max allowed time: {max_duration_hours} hours.")
    print("-" * 80)

    for epoch in range(num_epochs):
        epoch_start_time = time.time()

        # 检查是否超时
        current_total_time = time.time() - since
        if current_total_time > max_duration_seconds:
            print(f"\n[Phase {phase_id}] Timeout! Exceeded {max_duration_hours} hours. Stopping training early.")
            print(f"[Phase {phase_id}] Total runtime was ~{current_total_time / 3600:.2f} hours.")
            break

        model.train()
        running_loss = 0.0
        running_corrects = 0
        total_pixels = 0
        
        if epoch % 10 == 0 or epoch == num_epochs - 1:
            print(f"[Phase {phase_id}] Epoch {epoch + 1}/{num_epochs} | ", end="")
            sys.stdout.flush() # 强制刷新输出，确保提示立即出现

        for batch_idx, (inputs, labels, _) in enumerate(dataloader):
            inputs = inputs.to(device)
            labels = labels.squeeze(1).to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)
            total_pixels += labels.numel()

            # --- 采样逻辑 ---
            if batch_idx == 0 and (epoch + 1) % sample_interval == 0:
                sample_output = torch.argmax(outputs, dim=1).detach().cpu().numpy()
                sample_label = labels.detach().cpu().numpy()
                imgs_and_lbls = [(output, label) for output, label in zip(sample_output, sample_label)]
                logger.save_samples(
                    outputs_and_labels=imgs_and_lbls,
                    class_colors=[(255,0,0), (0,255,0), (0,0,255), (255, 255, 0), (255, 0, 255)],
                    sample_prefix=f"{batch_idx+1}"
                )
                print(f" [Sampled E{epoch+1}] ", end="")
                sys.stdout.flush()

        # 计算并打印当前 epoch 的统计信息
        epoch_duration = time.time() - epoch_start_time
        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_acc = running_corrects.double() / total_pixels

        # 简单的早停逻辑
        if last_loss is not None and abs(last_loss - epoch_loss) < loss_threshold:
            patience += 1
            if patience >= 5:
                print(f"Early stopping triggered after {patience} epochs with minimal loss change.")
                break
        else:
            patience = 0
        last_loss = epoch_loss

        # 记录日志
        logger.log_epoch(
            epoch + 1,
            epoch_loss, epoch_loss,
            epoch_duration, epoch_duration,
            optimizer.param_groups[0]['lr'],
            epoch_acc.item(), epoch_acc.item()
        )

        # 打印当前epoch的详细信息
        elapsed_time_str = time.strftime("%H:%M:%S", time.gmtime(time.time() - since))
        remaining_epochs = num_epochs - epoch - 1
        avg_epoch_time = (time.time() - since) / (epoch + 1)
        estimated_remaining_time = avg_epoch_time * remaining_epochs
        eta_str = time.strftime("%H:%M:%S", time.gmtime(estimated_remaining_time))

        if epoch % 10 == 0 or epoch == num_epochs - 1:
            print(f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
                  f"Time: {epoch_duration:.2f}s, ETA: {eta_str} (Total Elapsed: {elapsed_time_str})")

    inputs, labels, _ = next(iter(dataloader))
    inputs = inputs.to(device)
    labels = labels.squeeze(1).to(device)

    model.eval()

    outputs = model(inputs)
    sample_output = torch.argmax(outputs, dim=1).detach().cpu().numpy()
    sample_label = labels.detach().cpu().numpy()

    imgs_and_lbls = [(output, label) for output, label in zip(sample_output, sample_label)]
    logger.save_samples(
        outputs_and_labels=imgs_and_lbls,
        class_colors=[(255,0,0), (0,255,0), (0,0,255), (255, 255, 0), (255, 0, 255)],
        sample_prefix="phase_final",
    )
    print(f" [Sampled E_F] ")
    sys.stdout.flush()

    print("-" * 80)
    print(f"[Phase {phase_id}] Completed.")
    logger.finalize_phase_and_plot()
    
    logger.save_model_checkpoint(model=model, phase_id=phase_id)
    

In [ ]:
# --- 模型初始化设置 ---
INIT_MODEL_PATH = "" # 在此处填入.pth文件路径以加载预训练权重，留空则从头开始训练
if INIT_MODEL_PATH:
    model.load_state_dict(torch.load(INIT_MODEL_PATH, map_location=device))
    print(f"已从 '{INIT_MODEL_PATH}' 加载模型参数。")
else:
    print("未指定预训练模型，从头开始训练。")

# --- 必须定义类别数量 ---
NUM_CLASSES = 5 # 请将此值替换为您数据集的实际类别数量，例如PASCAL VOC有21类

# 初始化模型、损失函数、日志记录器等
model = model.to(device)
criterion = nn.CrossEntropyLoss() # CrossEntropyLoss 适用于整数标签 (N, C, H, W) -> target (N, H, W)

phases = [
    {'lr': 0.1, 'epochs': 20},
    {'lr': 0.01, 'epochs': 50},
    {'lr': 0.001, 'epochs': 100},
    {'lr': 0.0001, 'epochs': 150},
    {'lr': 0.00001, 'epochs': 200}
]

logger.log_initialization(
    model_name="MySemanticSegModel",
    dataset_name="MyDataset",
    train_size=len(train_loader.dataset),
    test_size=len(test_loader.dataset),
    optimizer_method="Adam"
)

for i, phase in enumerate(phases):
    optimizer = optim.Adam(model.parameters(), lr=phase['lr'])
    train_phase(
        model=model,
        dataloader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=phase['epochs'],
        num_classes=NUM_CLASSES, # 传递类别数量
        phase_id=i + 1,
        logger=logger,
        sample_interval=50 # 每50个epoch采样一次
    )

未指定预训练模型，从头开始训练。
[Phase 1] Starting training for 20 epochs with lr=0.100000...
[Phase 1] Max allowed time: 10 hours.
--------------------------------------------------------------------------------
[Phase 1] Epoch 1/20 | Loss: 0.9307, Acc: 0.6786, Time: 109.15s, ETA: 00:34:33 (Total Elapsed: 00:01:49)
[Phase 1] Epoch 11/20 | Loss: 0.7796, Acc: 0.7293, Time: 102.66s, ETA: 00:15:36 (Total Elapsed: 00:19:04)
[Phase 1] Epoch 20/20 | Loss: 0.6997, Acc: 0.7644, Time: 102.99s, ETA: 00:00:00 (Total Elapsed: 00:34:37)
 [Sampled E_F] 
--------------------------------------------------------------------------------
[Phase 1] Completed.
Phase 1 finalized. Plots saved to /home/jovyan/2026.3.17/resources/Logger/20260324_101040/Unet_Segmentation_Logger/phase_1_summary_plots.png.
[Model Save] 模型已保存至: /home/jovyan/2026.3.17/resources/Logger/20260324_101040/Unet_Segmentation_Logger/phase_1_checkpoints/model_phase_1_104520.pth
[Phase 2] Starting training for 50 epochs with lr=0.010000...
[Phase 2] Max al

In [ ]:
logger.finalize_run_and_create_final_plots()

# 显示一张或两张样本图片
from IPython.display import Image as IPImage, display
import glob

# 查找最新保存的样本图片
sample_dir = os.path.join(logger.log_root_dir, "phase_1_samples") # 可以修改为最后一个phase的目录
if os.path.exists(sample_dir):
    original_imgs = sorted(glob.glob(os.path.join(sample_dir, "*original.png")))
    mask_imgs = sorted(glob.glob(os.path.join(sample_dir, "*mask.png")))

    if original_imgs:
        print("Sample Original Image:")
        display(IPImage(filename=original_imgs[0]))
    if mask_imgs:
        print("Sample Mask Image:")
        display(IPImage(filename=mask_imgs[0]))
else:
    print(f"Sample directory {sample_dir} does not exist.")